# 🧠 Notebook 06 — LLM Narrative Engine
## BaliGuard — Early Warning System Krisis Pariwisata Bali

Membangun **LLM Narrative Engine** menggunakan **Claude API** yang mengubah output model menjadi laporan naratif otomatis Bahasa Indonesia.

**Input:** `data/final/predictions_final.csv`, `data/final/master_dataset_clean.parquet`  
**Output:** `src/narrative_engine.py`, `data/final/narratives_cache.json`

## 1. Import & Instalasi

In [10]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

try:
    from groq import Groq
    print('✅ groq SDK tersedia')
except ImportError:
    os.system('pip install groq -q')
    from groq import Groq
    print('✅ groq SDK berhasil diinstall')

print(f'Pandas: {pd.__version__}')


✅ groq SDK tersedia
Pandas: 2.3.3


## 2. Konfigurasi API Key

In [11]:
import os
from groq import Groq

# Set API key Groq — ganti string di bawah dengan key kamu
# Dapatkan gratis di: https://console.groq.com/keys

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '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')

if GROQ_API_KEY.startswith('gsk_GANTI'):
    print('⚠️  API key belum diset!')
    print('   Dapatkan gratis di: https://console.groq.com/keys')
else:
    client = Groq(api_key=GROQ_API_KEY)
    print('✅ Groq client siap')
    print(f'   Key: {GROQ_API_KEY[:12]}...')

✅ Groq client siap
   Key: gsk_xurGmatE...


## 3. Load Data & Model

In [12]:
predictions = pd.read_csv('data/final/predictions_final.csv')
print('✅ predictions_final:', predictions.shape)
print('   Kolom:', predictions.columns.tolist())
print()

master = pd.read_parquet('data/final/master_dataset_clean.parquet')
print('✅ master_dataset_clean:', master.shape)
print()

scaler   = joblib.load('data/models/scaler.pkl')
rf_model = joblib.load('data/models/model_random_forest.pkl')
le       = joblib.load('data/models/label_encoder.pkl')
print('✅ Model artifacts loaded')
print()

print('=== 6 BULAN TERAKHIR ===')
print(predictions.tail(6)[['month','wisman','crisis_score_100',
    'crisis_level','rf_predicted_level','rf_confidence','iso_anomaly']].to_string())

✅ predictions_final: (178, 18)
   Kolom: ['month', 'wisman', 'tpk_bintang', 'inflasi_processed', 'usd_idr_avg', 'avg_sentiment_monthly', 'bali_share_pct', 'wisman_zscore', 'crisis_score_100', 'crisis_level', 'rf_predicted_level', 'rf_confidence', 'prob_aman', 'prob_waspada', 'prob_siaga', 'prob_krisis', 'iso_anomaly', 'iso_score']

✅ master_dataset_clean: (192, 31)

✅ Model artifacts loaded

=== 6 BULAN TERAKHIR ===
       month  wisman  crisis_score_100 crisis_level rf_predicted_level  rf_confidence  iso_anomaly
172  2024-07  625665         30.000678      WASPADA            WASPADA       0.485802            0
173  2024-08  616641         40.289391      WASPADA            WASPADA       0.644891            0
174  2024-09  593909         42.322647      WASPADA            WASPADA       0.599900            0
175  2024-10  559911         44.035960      WASPADA            WASPADA       0.687239            0
176  2024-11  472900         54.510115        SIAGA              SIAGA       0.527166

## 4. Fungsi Build Context

In [13]:
def build_crisis_context(month_str: str, narratives_cache: dict = None) -> dict:
    """
    Bangun context dict lengkap dari satu bulan untuk dikirim ke LLM.
    Sekarang termasuk:
    - last_month_summary: ringkasan narasi bulan lalu (narrative consistency)
    - score_delta       : perubahan crisis score vs bulan lalu
    - dominant_factor   : faktor mana yang paling besar kontribusinya
    """
    pred_row = predictions[predictions['month'] == month_str]
    if len(pred_row) == 0:
        pred_row = predictions.tail(1)
        month_str = pred_row['month'].values[0]
    pred_row = pred_row.iloc[0]

    idx     = predictions[predictions['month'] == month_str].index[0]
    history = predictions.loc[max(0, idx-3):idx-1]

    # ── Context dasar ────────────────────────────────────────
    ctx = {
        'month'        : month_str,
        'crisis_score' : round(float(pred_row['crisis_score_100']), 1),
        'crisis_level' : str(pred_row.get('crisis_level', 'WASPADA')),
        'rf_predicted' : str(pred_row.get('rf_predicted_level', 'N/A')),
        'rf_confidence': round(float(pred_row.get('rf_confidence', 0)) * 100, 1),
        'is_anomaly'   : int(pred_row.get('iso_anomaly', 0)),
        'wisman'       : int(pred_row.get('wisman', 0)),
        'tpk_bintang'  : round(float(pred_row.get('tpk_bintang', 0)), 1),
        'inflasi'      : round(float(pred_row.get('inflasi_processed', 0)), 2),
        'usd_idr'      : round(float(pred_row.get('usd_idr_avg', 0)), 0),
        'sentiment'    : round(float(pred_row.get('avg_sentiment_monthly', 0)), 3),
        'prob_krisis'  : round(float(pred_row.get('prob_krisis', 0)) * 100, 1),
        'prob_siaga'   : round(float(pred_row.get('prob_siaga', 0)) * 100, 1),
        'bali_share'   : round(float(pred_row.get('bali_share_pct', 0)), 1),
        'wisman_zscore': round(float(pred_row.get('wisman_zscore', 0)), 2),
    }

    # ── Histori 3 bulan ──────────────────────────────────────
    if len(history) >= 2:
        avg3 = history['wisman'].mean()
        ctx['wisman_trend']  = 'naik' if ctx['wisman'] > avg3 else 'turun'
        ctx['avg_wisman_3m'] = round(avg3, 0)
        ctx['prev_levels']   = history['crisis_level'].tolist()
    else:
        ctx['wisman_trend']  = 'tidak tersedia'
        ctx['avg_wisman_3m'] = 0
        ctx['prev_levels']   = []

    # ── Score delta vs bulan lalu ─────────────────────────────
    if len(history) >= 1:
        prev_score = float(history.iloc[-1]['crisis_score_100'])
        ctx['score_delta'] = round(ctx['crisis_score'] - prev_score, 1)
        ctx['score_trend'] = 'MENINGKAT' if ctx['score_delta'] > 2 else (
                             'MENURUN'   if ctx['score_delta'] < -2 else 'STABIL')
    else:
        ctx['score_delta'] = 0.0
        ctx['score_trend'] = 'STABIL'

    # ── Dominant factor ───────────────────────────────────────
    factors = {
        'Kunjungan Wisatawan': abs(float(pred_row.get('wisman_zscore', 0))),
        'Tekanan Kurs'       : abs(float(pred_row.get('usd_volatility_3m', 0))) / 500.0,
        'Sentimen Negatif'   : float(pred_row.get('pct_negative_monthly', 0)) / 100.0
                               if 'pct_negative_monthly' in pred_row.index else 0.0,
    }
    ctx['dominant_factor'] = max(factors, key=factors.get)

    # ── Anomaly explanation ───────────────────────────────────
    zscore = ctx['wisman_zscore']
    if zscore <= -3:
        ctx['anomaly_explanation'] = (
            f'Z-score {zscore:.1f} → kunjungan {abs(zscore):.1f} std di bawah rata-rata 12 bulan. '
            'Ini adalah kejadian ekstrem (<0.1% probabilitas pada distribusi normal).')
    elif zscore <= -2:
        ctx['anomaly_explanation'] = (
            f'Z-score {zscore:.1f} → anomali signifikan, kunjungan jauh di bawah baseline historis.')
    else:
        ctx['anomaly_explanation'] = (
            f'Z-score {zscore:.1f} → kunjungan masih dalam rentang normal.')

    # ── Narrative consistency: ringkasan bulan lalu dari cache ─
    ctx['last_month_summary'] = ''
    if narratives_cache and len(history) >= 1:
        prev_month = history.iloc[-1]['month']
        if prev_month in narratives_cache:
            prev_narrative = narratives_cache[prev_month].get('narrative', '')
            # Ambil hanya kalimat pertama sebagai memory anchor
            ctx['last_month_summary'] = prev_narrative.split('.')[0] + '.' if prev_narrative else ''

    return ctx

# Test dengan bulan terbaru
test_month = predictions['month'].iloc[-1]
ctx_test   = build_crisis_context(test_month)
print(f'Context test ({test_month}):')
for k, v in ctx_test.items():
    print(f'  {k:22s}: {v}')


Context test (2024-12):
  month                 : 2024-12
  crisis_score          : 35.8
  crisis_level          : WASPADA
  rf_predicted          : WASPADA
  rf_confidence         : 66.3
  is_anomaly            : 0
  wisman                : 551100
  tpk_bintang           : 61.9
  inflasi               : 1.57
  usd_idr               : 16036.0
  sentiment             : 0.546
  prob_krisis           : 4.3
  prob_siaga            : 23.9
  bali_share            : 45.6
  wisman_zscore         : 0.35
  wisman_trend          : naik
  avg_wisman_3m         : 542240.0
  prev_levels           : ['WASPADA', 'WASPADA', 'SIAGA']
  score_delta           : -18.7
  score_trend           : MENURUN
  dominant_factor       : Kunjungan Wisatawan
  anomaly_explanation   : Z-score 0.3 → kunjungan masih dalam rentang normal.
  last_month_summary    : 


## 5. Fungsi Build Prompt

In [14]:
LEVEL_DESC = {
    'AMAN'    : 'kondisi pariwisata normal, tidak ada indikasi krisis',
    'WASPADA' : 'ada sinyal awal yang perlu dipantau',
    'SIAGA'   : 'tekanan signifikan pada sektor pariwisata',
    'KRISIS'  : 'krisis pariwisata yang membutuhkan respons segera'
}

def build_prompt(ctx: dict, report_type: str = 'summary') -> str:
    level_text = LEVEL_DESC.get(ctx['crisis_level'], ctx['crisis_level'])
    prev = ' → '.join(ctx['prev_levels']) if ctx['prev_levels'] else 'N/A'

    # ── Data block utama ──────────────────────────────────────
    data_block = (
        f"DATA PARIWISATA BALI — {ctx['month']}\n"
        f"Crisis Score: {ctx['crisis_score']}/100 → Level {ctx['crisis_level']} ({level_text})\n"
        f"Prediksi RF: {ctx['rf_predicted']} (confidence: {ctx['rf_confidence']}%)\n"
        f"Anomali: {'Ya' if ctx['is_anomaly'] else 'Tidak'} | "
        f"P(Krisis): {ctx['prob_krisis']}% | P(Siaga): {ctx['prob_siaga']}%\n"
        f"Wisman: {ctx['wisman']:,.0f} (trend: {ctx['wisman_trend']}, "
        f"avg 3bln: {int(ctx['avg_wisman_3m']):,.0f})\n"
        f"Pangsa Bali/Nasional: {ctx['bali_share']}% | Z-score: {ctx['wisman_zscore']}\n"
        f"TPK Hotel: {ctx['tpk_bintang']}% | USD/IDR: Rp {int(ctx['usd_idr']):,}\n"
        f"Inflasi: {ctx['inflasi']}% | Sentimen: {ctx['sentiment']}\n"
        f"Histori 3 bulan: {prev}\n"
    )

    # ── Comparative & causal block (BARU) ─────────────────────
    comparative_block = (
        f"\nANALISIS KOMPARATIF:\n"
        f"Tren Crisis Score: {ctx['score_trend']} "
        f"({ctx['score_delta']:+.1f} poin vs bulan lalu)\n"
        f"Faktor dominan: {ctx['dominant_factor']}\n"
        f"Penjelasan anomali: {ctx['anomaly_explanation']}\n"
    )

    # ── Narrative consistency block (BARU) ────────────────────
    continuity_block = ''
    if ctx.get('last_month_summary'):
        continuity_block = (
            f"\nKONTEKS BULAN LALU:\n"
            f"{ctx['last_month_summary']}\n"
            f"(Gunakan ini sebagai referensi perbandingan — bukan diulang kembali)\n"
        )

    full_data = data_block + comparative_block + continuity_block

    if report_type == 'summary':
        return (
            "Kamu adalah analis sistem BaliGuard (early warning pariwisata Bali).\n"
            + full_data
            + f"\nBuat ringkasan kondisi pariwisata Bali bulan {ctx['month']} "
            "dalam 2-3 kalimat Bahasa Indonesia. Sertakan: "
            "(1) status & skor, (2) satu angka kunci sebagai bukti, "
            "(3) arah tren vs bulan lalu. Tajam, berbasis data, cocok untuk KPI card."
        )
    elif report_type == 'alert':
        return (
            "Kamu adalah sistem BaliGuard. Kondisi kritis terdeteksi.\n"
            + full_data
            + "\nBuat PERINGATAN DARURAT Bahasa Indonesia (max 120 kata):\n"
            "1. Status level dan crisis score\n"
            "2. 3 indikator kritis yang MENYEBABKAN kondisi ini (sebab-akibat, bukan hanya angka)\n"
            "3. 1 rekomendasi tindakan segera yang konkret dan bisa dieksekusi dalam 7 hari\n"
            "Tone profesional, urgen, berbasis data."
        )
    else:
        return (
            "Kamu adalah analis BaliGuard.\n"
            + full_data
            + "\nBuat laporan bulanan Bahasa Indonesia dengan REASONING yang kuat:\n"
            "1. **Ringkasan Eksekutif** (2-3 kalimat): status, skor, perbandingan bulan lalu\n"
            "2. **Analisis Indikator** (3-4 kalimat): apa yang ditunjukkan angka-angka, "
            "mengapa setiap indikator PENTING untuk disimpulkan\n"
            "3. **Analisis Kausal** (2-3 kalimat): APA yang menyebabkan kondisi ini — "
            "bukan hanya deskripsi angka, tapi hubungan sebab-akibat antar indikator\n"
            "4. **Rekomendasi** (3 poin konkret dengan target waktu): "
            "setiap poin harus spesifik, actionable, dan berbasis data di atas\n"
            "Setiap klaim harus didukung minimal satu angka dari data."
        )

print(f'Summary prompt : {len(build_prompt(ctx_test, "summary"))} karakter')
print(f'Alert prompt   : {len(build_prompt(ctx_test, "alert"))} karakter')
print(f'Monthly prompt : {len(build_prompt(ctx_test, "monthly"))} karakter')
print()
print('=== SAMPLE MONTHLY PROMPT (200 char) ===')
print(build_prompt(ctx_test, 'monthly')[:200])


Summary prompt : 887 karakter
Alert prompt   : 942 karakter
Monthly prompt : 1219 karakter

=== SAMPLE MONTHLY PROMPT (200 char) ===
Kamu adalah analis BaliGuard.
DATA PARIWISATA BALI — 2024-12
Crisis Score: 35.8/100 → Level WASPADA (ada sinyal awal yang perlu dipantau)
Prediksi RF: WASPADA (confidence: 66.3%)
Anomali: Tidak | P(Kr


## 6. Fungsi Generate Narasi (Claude API)

In [15]:
def generate_narrative(month_str: str, report_type: str = 'summary',
                       api_key: str = None) -> dict:
    if api_key is None:
        api_key = os.getenv('GROQ_API_KEY', '')
    if not api_key or api_key.startswith('gsk_GANTI'):
        return {'success'  : False,
                'narrative': '[API key belum dikonfigurasi. Set GROQ_API_KEY.]',
                'error'    : 'No API key'}
    try:
        ctx      = build_crisis_context(month_str)
        prompt   = build_prompt(ctx, report_type)
        client   = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model = 'llama-3.3-70b-versatile',
            messages    = [{'role': 'user', 'content': prompt}],
            temperature = 0.7,
            max_tokens  = 1024
        )
        return {
            'success'      : True,
            'narrative'    : response.choices[0].message.content,
            'tokens'       : response.usage.prompt_tokens + response.usage.completion_tokens,
            'month'        : month_str,
            'crisis_level' : ctx['crisis_level'],
            'report_type'  : report_type,
            'crisis_score' : ctx['crisis_score'],
        }
    except Exception as e:
        return {'success': False, 'narrative': f'[Error: {e}]', 'error': str(e)}

# Test
print(f'Testing generate untuk bulan: {test_month}')
result = generate_narrative(test_month, 'summary', GROQ_API_KEY)
print(f'Success: {result["success"]}')
if result['success']:
    print(f'Tokens : {result["tokens"]}')
    print()
    print('=== NARASI ===')
    print(result['narrative'])
else:
    print(f'Error  : {result.get("error")}')


Testing generate untuk bulan: 2024-12
Success: True
Tokens : 497

=== NARASI ===
Kondisi pariwisata Bali pada bulan Desember 2024 berstatus WASPADA dengan skor 35,8/100. Dengan jumlah wisman mencapai 551.100, menunjukkan tren kenaikan, kondisi ini menunjukkan peningkatan dibandingkan bulan sebelumnya. Skor Crisis Score menurun sebesar 18,7 poin, mengindikasikan perbaikan kondisi pariwisata Bali.


## 7. Batch Generate — Semua Periode SIAGA & KRISIS

In [16]:
crisis_months = predictions[
    predictions['crisis_level'].isin(['KRISIS', 'SIAGA'])
].sort_values('crisis_score_100', ascending=False)

print(f'Bulan KRISIS/SIAGA: {len(crisis_months)}')
narratives_cache = {}

if not GROQ_API_KEY.startswith('gsk_GANTI'):
    top_months = crisis_months.head(10)['month'].tolist()
    print(f'Generating {len(top_months)} narasi...')
    print()

    for i, month in enumerate(top_months, 1):
        level = predictions[predictions['month']==month]['crisis_level'].values[0]
        rtype = 'alert' if level == 'KRISIS' else 'monthly'
        print(f'  [{i:2d}/{len(top_months)}] {month} ({level})...', end='', flush=True)
        result = generate_narrative(month, rtype, GROQ_API_KEY)
        if result['success']:
            narratives_cache[month] = result
            print(f' ✅ {result["tokens"]} tokens')
        else:
            print(f' ❌ {result["error"]}')

    os.makedirs('data/final', exist_ok=True)
    with open('data/final/narratives_cache.json', 'w', encoding='utf-8') as f:
        json.dump(narratives_cache, f, ensure_ascii=False, indent=2)
    print(f'\n✅ narratives_cache.json disimpan ({len(narratives_cache)} narasi)')
else:
    print('⚠️  API key belum diset — skip batch generation')


Bulan KRISIS/SIAGA: 36
Generating 10 narasi...

  [ 1/10] 2020-03 (KRISIS)... ✅ 602 tokens
  [ 2/10] 2020-04 (KRISIS)... ✅ 626 tokens
  [ 3/10] 2020-07 (KRISIS)... ✅ 553 tokens
  [ 4/10] 2020-05 (KRISIS)... ✅ 563 tokens
  [ 5/10] 2020-02 (KRISIS)... ✅ 529 tokens
  [ 6/10] 2015-11 (SIAGA)... ✅ 1203 tokens
  [ 7/10] 2020-08 (SIAGA)... ✅ 1255 tokens
  [ 8/10] 2021-03 (SIAGA)... ✅ 1117 tokens
  [ 9/10] 2020-09 (SIAGA)... ✅ 1123 tokens
  [10/10] 2021-07 (SIAGA)... ✅ 1123 tokens

✅ narratives_cache.json disimpan (10 narasi)


## 8. Export `src/narrative_engine.py`

In [17]:
LEVEL_DESC = {
    'AMAN'    : 'kondisi pariwisata normal, tidak ada indikasi krisis',
    'WASPADA' : 'ada sinyal awal yang perlu dipantau',
    'SIAGA'   : 'tekanan signifikan pada sektor pariwisata',
    'KRISIS'  : 'krisis pariwisata yang membutuhkan respons segera'
}

def build_prompt(ctx: dict, report_type: str = 'summary') -> str:
    level_text = LEVEL_DESC.get(ctx['crisis_level'], ctx['crisis_level'])
    prev = ' → '.join(ctx['prev_levels']) if ctx['prev_levels'] else 'N/A'

    # ── Data block utama ──────────────────────────────────────
    data_block = (
        f"DATA PARIWISATA BALI — {ctx['month']}\n"
        f"Crisis Score: {ctx['crisis_score']}/100 → Level {ctx['crisis_level']} ({level_text})\n"
        f"Prediksi RF: {ctx['rf_predicted']} (confidence: {ctx['rf_confidence']}%)\n"
        f"Anomali: {'Ya' if ctx['is_anomaly'] else 'Tidak'} | "
        f"P(Krisis): {ctx['prob_krisis']}% | P(Siaga): {ctx['prob_siaga']}%\n"
        f"Wisman: {ctx['wisman']:,.0f} (trend: {ctx['wisman_trend']}, "
        f"avg 3bln: {int(ctx['avg_wisman_3m']):,.0f})\n"
        f"Pangsa Bali/Nasional: {ctx['bali_share']}% | Z-score: {ctx['wisman_zscore']}\n"
        f"TPK Hotel: {ctx['tpk_bintang']}% | USD/IDR: Rp {int(ctx['usd_idr']):,}\n"
        f"Inflasi: {ctx['inflasi']}% | Sentimen: {ctx['sentiment']}\n"
        f"Histori 3 bulan: {prev}\n"
    )

    # ── Comparative & causal block (BARU) ─────────────────────
    comparative_block = (
        f"\nANALISIS KOMPARATIF:\n"
        f"Tren Crisis Score: {ctx['score_trend']} "
        f"({ctx['score_delta']:+.1f} poin vs bulan lalu)\n"
        f"Faktor dominan: {ctx['dominant_factor']}\n"
        f"Penjelasan anomali: {ctx['anomaly_explanation']}\n"
    )

    # ── Narrative consistency block (BARU) ────────────────────
    continuity_block = ''
    if ctx.get('last_month_summary'):
        continuity_block = (
            f"\nKONTEKS BULAN LALU:\n"
            f"{ctx['last_month_summary']}\n"
            f"(Gunakan ini sebagai referensi perbandingan — bukan diulang kembali)\n"
        )

    full_data = data_block + comparative_block + continuity_block

    if report_type == 'summary':
        return (
            "Kamu adalah analis sistem BaliGuard (early warning pariwisata Bali).\n"
            + full_data
            + f"\nBuat ringkasan kondisi pariwisata Bali bulan {ctx['month']} "
            "dalam 2-3 kalimat Bahasa Indonesia. Sertakan: "
            "(1) status & skor, (2) satu angka kunci sebagai bukti, "
            "(3) arah tren vs bulan lalu. Tajam, berbasis data, cocok untuk KPI card."
        )
    elif report_type == 'alert':
        return (
            "Kamu adalah sistem BaliGuard. Kondisi kritis terdeteksi.\n"
            + full_data
            + "\nBuat PERINGATAN DARURAT Bahasa Indonesia (max 120 kata):\n"
            "1. Status level dan crisis score\n"
            "2. 3 indikator kritis yang MENYEBABKAN kondisi ini (sebab-akibat, bukan hanya angka)\n"
            "3. 1 rekomendasi tindakan segera yang konkret dan bisa dieksekusi dalam 7 hari\n"
            "Tone profesional, urgen, berbasis data."
        )
    else:
        return (
            "Kamu adalah analis BaliGuard.\n"
            + full_data
            + "\nBuat laporan bulanan Bahasa Indonesia dengan REASONING yang kuat:\n"
            "1. **Ringkasan Eksekutif** (2-3 kalimat): status, skor, perbandingan bulan lalu\n"
            "2. **Analisis Indikator** (3-4 kalimat): apa yang ditunjukkan angka-angka, "
            "mengapa setiap indikator PENTING untuk disimpulkan\n"
            "3. **Analisis Kausal** (2-3 kalimat): APA yang menyebabkan kondisi ini — "
            "bukan hanya deskripsi angka, tapi hubungan sebab-akibat antar indikator\n"
            "4. **Rekomendasi** (3 poin konkret dengan target waktu): "
            "setiap poin harus spesifik, actionable, dan berbasis data di atas\n"
            "Setiap klaim harus didukung minimal satu angka dari data."
        )

print(f'Summary prompt : {len(build_prompt(ctx_test, "summary"))} karakter')
print(f'Alert prompt   : {len(build_prompt(ctx_test, "alert"))} karakter')
print(f'Monthly prompt : {len(build_prompt(ctx_test, "monthly"))} karakter')
print()
print('=== SAMPLE MONTHLY PROMPT (200 char) ===')
print(build_prompt(ctx_test, 'monthly')[:200])


Summary prompt : 887 karakter
Alert prompt   : 942 karakter
Monthly prompt : 1219 karakter

=== SAMPLE MONTHLY PROMPT (200 char) ===
Kamu adalah analis BaliGuard.
DATA PARIWISATA BALI — 2024-12
Crisis Score: 35.8/100 → Level WASPADA (ada sinyal awal yang perlu dipantau)
Prediksi RF: WASPADA (confidence: 66.3%)
Anomali: Tidak | P(Kr


## 9. Checklist File

In [18]:
print('=' * 55)
print('  BALIGUARD — RINGKASAN NB06')
print('=' * 55)
print()
required = [
    'data/final/master_dataset_clean.parquet',
    'data/final/predictions_final.csv',
    'data/final/narratives_cache.json',
    'data/models/model_random_forest.pkl',
    'data/models/model_isolation_forest.pkl',
    'data/models/scaler.pkl',
    'data/models/label_encoder.pkl',
    'src/narrative_engine.py',
]
for f in required:
    status = '✅' if os.path.exists(f) else '❌ belum ada'
    print(f'   {status} {f}')
print()
print('Jalankan dashboard:')
print('  pip install streamlit plotly anthropic pyarrow joblib')
print('  streamlit run dashboard.py')
print('=' * 55)

  BALIGUARD — RINGKASAN NB06

   ✅ data/final/master_dataset_clean.parquet
   ✅ data/final/predictions_final.csv
   ✅ data/final/narratives_cache.json
   ✅ data/models/model_random_forest.pkl
   ✅ data/models/model_isolation_forest.pkl
   ✅ data/models/scaler.pkl
   ✅ data/models/label_encoder.pkl
   ✅ src/narrative_engine.py

Jalankan dashboard:
  pip install streamlit plotly anthropic pyarrow joblib
  streamlit run dashboard.py
